# Parcel-state crop classifier — augmented training v2
Trains a separate `closed_box` / `open_box` MobileNetV3-Small classifier from the manually reviewed parcel crops. This revision uses task-specific phone-image augmentation and keeps the original validation split out of threshold selection. The generic parcel YOLO remains the localization stage.

In [ ]:
import csv, json, random, shutil, zipfile
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, models, transforms
from google.colab import files
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
print('Torch:', torch.__version__)

In [ ]:
uploaded = files.upload()  # Choose parcel_state_classification_dataset.zip
zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
assert zip_name, 'Upload the reviewed classification dataset ZIP.'
extract_root = Path('/content/parcel_state_data')
if extract_root.exists(): shutil.rmtree(extract_root)
with zipfile.ZipFile(zip_name) as archive: archive.extractall(extract_root)
roots = [p for p in extract_root.rglob('train') if (p.parent / 'valid').is_dir()]
assert len(roots) == 1, f'Expected one dataset root; found {roots}'
data_root = roots[0].parent
print('Dataset:', data_root)

In [ ]:
IMAGE_SIZE = 224
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    # Phone-camera geometry without transformations that change open/closed semantics.
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.82, 1.0), ratio=(0.85, 1.15)),
    transforms.RandomHorizontalFlip(p=0.5),  # Never use vertical flips for this task.
    transforms.RandomApply([transforms.RandomPerspective(distortion_scale=0.15, p=1.0)], p=0.30),
    transforms.RandomAffine(degrees=8, translate=(0.05, 0.05), scale=(0.92, 1.08), shear=3),
    transforms.RandomApply([transforms.ColorJitter(brightness=.30, contrast=.30, saturation=.20, hue=.03)], p=.75),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2))], p=.15),
    transforms.RandomGrayscale(p=.04),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=.12, scale=(.01, .05), ratio=(.5, 2.0), value='random'),
])
valid_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
train_augmented = datasets.ImageFolder(data_root / 'train', transform=train_tf)
train_evaluation = datasets.ImageFolder(data_root / 'train', transform=valid_tf)
valid_ds = datasets.ImageFolder(data_root / 'valid', transform=valid_tf)
assert train_augmented.class_to_idx == {'closed_box': 0, 'open_box': 1}, train_augmented.class_to_idx
assert valid_ds.class_to_idx == train_augmented.class_to_idx
# Hold calibration images out of fitting. The original validation split remains untouched for final evaluation.
rng = np.random.default_rng(SEED); fit_indices, calibration_indices = [], []
targets = np.asarray(train_augmented.targets)
for class_id in range(2):
    indices = np.flatnonzero(targets == class_id); rng.shuffle(indices)
    n_calibration = max(8, round(0.15 * len(indices)))
    calibration_indices.extend(indices[:n_calibration]); fit_indices.extend(indices[n_calibration:])
train_ds = Subset(train_augmented, fit_indices)
calibration_ds = Subset(train_evaluation, calibration_indices)
fit_targets = targets[fit_indices]; counts = np.bincount(fit_targets, minlength=2)
sample_weights = torch.as_tensor([1.0 / counts[label] for label in fit_targets], dtype=torch.double)
sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler, num_workers=2, pin_memory=True)
calibration_loader = DataLoader(calibration_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)
print('Classes:', train_augmented.class_to_idx)
print('Fit:', counts.tolist(), 'calibration:', np.bincount(targets[calibration_indices], minlength=2).tolist(),
      'final validation:', np.bincount(valid_ds.targets, minlength=2).tolist())

In [ ]:
def metrics_at_threshold(labels, probs, threshold):
    labels, pred = np.asarray(labels), (np.asarray(probs) >= threshold).astype(int)
    cm = np.zeros((2, 2), dtype=int)
    for actual, predicted in zip(labels, pred): cm[actual, predicted] += 1
    per_class = []
    for c in range(2):
        tp, fp, fn = cm[c,c], cm[:,c].sum()-cm[c,c], cm[c,:].sum()-cm[c,c]
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2*precision*recall/(precision+recall) if precision+recall else 0.0
        per_class.append({'precision': precision, 'recall': recall, 'f1': f1, 'support': int(cm[c,:].sum())})
    return float(np.mean([m['f1'] for m in per_class])), cm, per_class

def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training); total_loss = 0.; labels_all, probs_all = [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if training: optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(training):
            logits = model(images); loss = criterion(logits, labels)
            if training: loss.backward(); optimizer.step()
        total_loss += loss.item() * len(labels)
        labels_all.extend(labels.cpu().tolist())
        probs_all.extend(torch.softmax(logits, 1)[:,1].detach().cpu().tolist())
    return total_loss / len(loader.dataset), labels_all, probs_all

In [ ]:
EPOCHS, PATIENCE, HEAD_ONLY_EPOCHS = 25, 6, 4
model = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1)
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, 2)
model.to(device)
for parameter in model.features.parameters(): parameter.requires_grad = False
criterion = nn.CrossEntropyLoss(label_smoothing=.05)
optimizer = torch.optim.AdamW(model.classifier.parameters(), lr=5e-4, weight_decay=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=.3)
history, best_f1, stale = [], -1., 0
checkpoint_path = Path('/content/parcel_state_classifier_best.pt')
for epoch in range(1, EPOCHS + 1):
    if epoch == HEAD_ONLY_EPOCHS + 1:
        for parameter in model.features[-3:].parameters(): parameter.requires_grad = True
        optimizer = torch.optim.AdamW([
            {'params': model.classifier.parameters(), 'lr': 2e-4},
            {'params': model.features[-3:].parameters(), 'lr': 2e-5},
        ], weight_decay=3e-4)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=2, factor=.3)
        print('Unfroze the final three MobileNet feature blocks.')
    train_loss, train_y, train_p = run_epoch(model, train_loader, criterion, optimizer)
    calibration_loss, calibration_y, calibration_p = run_epoch(model, calibration_loader, criterion)
    train_f1, _, _ = metrics_at_threshold(train_y, train_p, .5)
    threshold_scores = [(metrics_at_threshold(calibration_y, calibration_p, float(t))[0], float(t)) for t in np.linspace(.10, .90, 81)]
    calibration_f1, threshold = max(threshold_scores, key=lambda item: (item[0], -abs(item[1]-.5)))
    history.append({'epoch': epoch, 'train_loss': train_loss, 'calibration_loss': calibration_loss,
                    'train_macro_f1': train_f1, 'calibration_macro_f1': calibration_f1, 'threshold': threshold})
    print(f'{epoch:02d} train_loss={train_loss:.4f} cal_loss={calibration_loss:.4f} cal_macro_f1={calibration_f1:.4f} threshold={threshold:.2f}')
    scheduler.step(calibration_f1)
    if calibration_f1 > best_f1:
        best_f1, stale = calibration_f1, 0
        torch.save({'model_state_dict': model.state_dict(), 'architecture': 'mobilenet_v3_small',
                    'class_to_idx': train_augmented.class_to_idx, 'image_size': IMAGE_SIZE,
                    'normalization': {'mean': MEAN, 'std': STD}, 'epoch': epoch,
                    'open_box_threshold': threshold, 'augmentation_version': 'phone_box_v2'}, checkpoint_path)
    else:
        stale += 1
        if stale >= PATIENCE: print('Early stopping'); break

In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
valid_loss, valid_y, valid_p = run_epoch(model, valid_loader, criterion)
threshold = float(checkpoint['open_box_threshold'])  # Fixed before looking at final validation.
macro_f1, cm, per_class = metrics_at_threshold(valid_y, valid_p, threshold)
checkpoint['validation_macro_f1'] = macro_f1
torch.save(checkpoint, checkpoint_path)
names = ['closed_box', 'open_box']
metrics = {'threshold': threshold, 'macro_f1': macro_f1, 'confusion_matrix': cm.tolist(),
           'per_class': {name: per_class[i] for i, name in enumerate(names)},
           'validation_samples': len(valid_ds), 'best_epoch': checkpoint['epoch'],
           'augmentation_version': checkpoint['augmentation_version'],
           'evaluation_protocol': 'threshold selected on train calibration subset; original validation evaluated once'}
print(json.dumps(metrics, indent=2))
results = Path('/content/parcel_state_classifier_results'); results.mkdir(exist_ok=True)
with (results/'metrics.json').open('w') as f: json.dump(metrics, f, indent=2)
with (results/'history.csv').open('w', newline='') as f:
    writer=csv.DictWriter(f, fieldnames=history[0]); writer.writeheader(); writer.writerows(history)
fig, ax = plt.subplots(figsize=(5,4)); im=ax.imshow(cm, cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j, i, cm[i,j], ha='center', va='center')
ax.set(xticks=[0,1], yticks=[0,1], xticklabels=names, yticklabels=names, xlabel='Predicted', ylabel='Actual', title='Validation confusion matrix')
fig.colorbar(im, ax=ax); fig.tight_layout(); fig.savefig(results/'confusion_matrix.png', dpi=160); plt.show()
shutil.copy2(checkpoint_path, results/checkpoint_path.name)
results_zip = shutil.make_archive('/content/parcel_state_classifier_training_results', 'zip', results)

In [ ]:
files.download(str(checkpoint_path))
files.download(results_zip)
print('Return both downloaded files to colab/parcel_state_classifier_training/')